In [17]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# 通过load_dotenv()将.env中的变量加载为环境变量
# override=True表示：无论你当前的操作系统、终端或者虚拟环境中是否已经存在同名的环境变量，
# 都会强行用 .env 文件里写的值去覆盖它
load_dotenv(override=True)


# 从环境变量读取配置
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")


# 创建LLM
model = init_chat_model(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    model="deepseek:deepseek-flash" # 冒号前面直定模型厂商，冒号后面指定模型
)

In [2]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


checkpointer = InMemorySaver()  #1、创建了内存级的记忆存储


agent = create_agent(
    model=model,
    tools=[],
    checkpointer=checkpointer  #2、让agent具备了存储的能力
)

# 3、同一个thread_id共享记忆的。
config = {
    "configurable" : {
        "thread_id" : "1"
    }
}

print("\n第一轮对话：")
response1 = agent.invoke({
    "messages": [HumanMessage("我叫张三")]},
    config=config   # 4、传入invoke()当中
)
print(f"Agent: {response1['messages'][-1].content}")

print("\n第二轮对话：")
response2 = agent.invoke({
    "messages": [HumanMessage("我叫什么？")]},
    config=config
)
print(f"Agent: {response2['messages'][-1].content}")


第一轮对话：
Agent: 你好，张三！很高兴认识你。有什么我可以帮你的吗？

第二轮对话：
Agent: 你叫张三。


In [3]:
print("\n第三轮对话：")
response3 = agent.invoke({
    "messages": [HumanMessage("我刚才问了什么问题？")]},
    config=config
)
print(f"Agent: {response3['messages'][-1].content}")


第三轮对话：
Agent: 你刚才问的是：“我叫什么？”


体会不同session不共享信息

In [6]:
config2 = {
    "configurable" : {
        "thread_id" : "2"
    }
}

print("\n第四轮对话：")
response4 = agent.invoke({
    "messages": [HumanMessage("我叫什么？")]},
    config=config2,
    checkpointer=checkpointer,
)
print(f"Agent: {response4['messages'][-1].content}")


print("\n第五轮对话：")
response5 = agent.invoke({
    "messages": [HumanMessage("前面我问了你什么问题")]},
    config=config2,
    checkpointer=checkpointer,
)
print(f"Agent: {response5['messages'][-1].content}")


第四轮对话：
Agent: 你还没告诉我你的名字，所以我不知道。你告诉我后，我就能在这次对话里称呼你。

第五轮对话：
Agent: 你前面连续问了三次：**“我叫什么？”**
